# Typicality analysis across drift strength

Evaluates FA/Hit/d' vs drift (`alpha`) for typical vs atypical items.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from functools import partial
from utls.runners_v2 import run_experiment_scores_prior
from utls.sigma_fitting import _compute_auroc_upper_envelope
from utls.analysis_helpers import auroc_to_dprime



## Typicality definition
Here we use score-norm as a proxy (`lower ||score(x)||` = more typical).


In [ ]:
with torch.no_grad():
    x = torch.as_tensor(X, dtype=torch.float32, device=next(score_fn.parameters()).device)
    s = score_fn.forward(x).reshape(len(X), -1)
    typ = torch.linalg.norm(s, dim=1).cpu().numpy()
median_typ = np.median(typ)
is_typical = typ <= median_typ



In [ ]:
# Sweep alpha and collect itemwise scores
alphas = np.geomspace(1e-4, 2e-1, 8)
results = []
for a in alphas:
    out = run_experiment_scores_prior(
        sigma0=sigma0_fitted,
        noise_mode=noise_mode,
        metric=metric,
        X0=X,
        name_to_idx=name_to_idx,
        experiment_list=experiment_list,
        score_model=score_fn,
        drift_step_size=float(a),
        return_item_scores=True,
    )
    results.append((a, out))



In [ ]:
# Aggregate by typical / atypical groups (fill with your item-index mapping)
# Compute group-wise FA rates, hit rates, and d' per alpha.

